# Challenge 02 — Analítica Multidimensional
### TechLogistics S.A. · Metodología CRISP-DM · EAFIT — Maestría en Ciencia de los Datos

Este notebook resuelve, de principio a fin, el reto de **analítica multidimensional**
sobre dos dominios operativos de TechLogistics:

- **Agroindustria / Clima** (`agro_*`) — red de sensores tipo *mesh* en el Oriente Antioqueño.
- **Energía / Economía** (`ener_*`) — red de *despacho* eléctrico.

Se cubren cuatro fases CRISP-DM: (1) *Data Understanding* + geo-visualización,
(2) procesamiento de señales y filtrado, (3) grafos y topología de red, y
(4) modelado y decisiones de negocio.

---
## Supuestos metodológicos (declarados explícitamente)

1. **Existencia de señal *clean*.** El brief asumía que *solo* se entregaron los archivos
   `*_noise`. Sin embargo, los archivos `agro_clean.csv` y `ener_clean.csv` **sí existen**
   en `data/`. Por rigor usamos esa **señal limpia real como referencia (ground truth)**
   para SNR y RMSE, en lugar de estimarla. Aun así construimos la versión *denoised*
   (Butterworth / media móvil) para demostrar el pipeline de filtrado y para el caso en
   que la referencia no estuviera disponible.
2. **Sin timestamp.** Los CSV no traen columna temporal. Creamos un `DatetimeIndex`
   **sintético horario** (`freq='h'`, inicio fijo `2023-01-01`) para todo el análisis de
   series de tiempo. Es un supuesto: asume muestreo equiespaciado de 1 registro/hora.
3. **Reproducibilidad.** Fijamos la semilla de NumPy (`seed=42`).
4. **Diccionario de variables.** Interpretamos cada variable según el diccionario provisto
   (no se inventan significados).

### Configuración del entorno y utilidades reutilizables (`src/`)

In [ ]:
import sys
from pathlib import Path

# --- Localizar la raíz del proyecto de forma robusta (busca requirements.txt) ---
def find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "requirements.txt").exists() and (cand / "src").exists():
            return cand
    return start.resolve().parent

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import io_utils, stationarity, signal_utils, graph_utils, viz_utils

io_utils.set_seeds(42)               # reproducibilidad
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Librerías cargadas correctamente.")

### Carga de datos y creación del índice temporal sintético

Cargamos los cuatro archivos y adjuntamos el índice horario sintético. Recordar que
`clean` es la **referencia real** y `noise` es la señal observada con ruido inyectado
(SNR objetivo del reto en el rango 5–12 dB).

In [ ]:
agro_clean, agro_noise = io_utils.load_pair(DATA, "agro")
ener_clean, ener_noise = io_utils.load_pair(DATA, "ener")

AGRO_COLS = io_utils.value_columns(agro_noise, "Agro")
ENER_COLS = io_utils.value_columns(ener_noise, "Ener")

print("agro_noise:", agro_noise.shape, "| ener_noise:", ener_noise.shape)
print("Índice temporal sintético:", agro_noise.index.min(), "->", agro_noise.index.max())
agro_noise.head(3)

Resumen estadístico rápido para entender escalas y detectar anomalías obvias.

In [ ]:
display(agro_noise[AGRO_COLS].describe().T[["mean", "std", "min", "max"]])
display(ener_noise[ENER_COLS].describe().T[["mean", "std", "min", "max"]])

---
# FASE 1 — Data Understanding y Geo-Visualización

## T1 · Geo-visualización de sensores (Plotly `scatter_mapbox`)

Ubicamos los sensores agro en el mapa del Oriente Antioqueño. **Color = NDVI (`Agro_5`)**
(verde = vegetación sana, rojo = baja) y **tamaño = Humedad del suelo (`Agro_1`)**.
Buscamos *clustering espacial*: zonas donde la biomasa/NDVI es consistentemente baja,
candidatas a intervención. Usamos el estilo `open-street-map` (no requiere token Mapbox).

In [ ]:
fig_geo = viz_utils.geo_sensor_map(
    agro_noise, color="Agro_5", size="Agro_1",
    color_label="NDVI (Agro_5)", size_label="Humedad (Agro_1)",
    title="T1 · Sensores agro — NDVI (color) y Humedad (tamaño)",
)
try:
    viz_utils.save_plotly(fig_geo, FIGS / "t1_geo_ndvi.png")
    print("Figura exportada: figures/t1_geo_ndvi.png")
except Exception as e:
    print("No se pudo exportar PNG con kaleido, se guarda fallback matplotlib:", e)
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(agro_noise["Longitude"], agro_noise["Latitude"],
                     c=agro_noise["Agro_5"], s=20, cmap="RdYlGn")
    plt.colorbar(sc, label="NDVI"); plt.xlabel("Longitud"); plt.ylabel("Latitud")
    plt.title("T1 · Sensores agro (fallback)"); viz_utils.savefig(FIGS / "t1_geo_ndvi.png")
    plt.close()
fig_geo.show()

**Detección de clustering espacial de bajo NDVI.** Marcamos el cuartil inferior de NDVI
y verificamos si esos sensores se concentran geográficamente (usamos la mediana de
lat/lon como partición simple y reportamos la fracción de sensores de bajo NDVI por zona).

In [ ]:
q1_ndvi = agro_noise["Agro_5"].quantile(0.25)
low = agro_noise[agro_noise["Agro_5"] <= q1_ndvi]
lat_med, lon_med = agro_noise["Latitude"].median(), agro_noise["Longitude"].median()

def zone(row):
    ns = "N" if row["Latitude"] >= lat_med else "S"
    ew = "E" if row["Longitude"] >= lon_med else "O"
    return ns + ew

agro_noise["_zona"] = agro_noise.apply(zone, axis=1)
low = low.assign(_zona=low.apply(zone, axis=1))
share = (low["_zona"].value_counts(normalize=True) * 100).round(1)
print(f"Umbral NDVI (Q1) = {q1_ndvi:.3f}. Distribución de sensores de BAJO NDVI por zona (%):")
print(share.to_string())
print("\\nInterpretación: una zona que concentra desproporcionadamente el bajo NDVI "
      "sugiere clustering espacial de estrés vegetal (no ruido aleatorio).")

## T2 · Estacionariedad de las series de energía (ADF) + estadísticos móviles

Aplicamos el test **Augmented Dickey-Fuller** a las 10 series de energía. H0: existe raíz
unitaria (serie **no** estacionaria). Si `p < 0.05` rechazamos H0 → estacionaria.
Esperamos, según el diccionario: `Ener_1-3` correlacionadas, `Ener_5-7` **no**
estacionarias (macro), `Ener_8-10` estacionarias (calidad de potencia).

In [ ]:
adf_ener = stationarity.adf_table(ener_noise, ENER_COLS, io_utils.ENER_NAMES)
display(adf_ener)

### ¿`Ener_5` (Costo del Gas) es *Drift* o *Random Walk*?

Para una serie **no estacionaria** aplicamos ventana móvil de **50 registros** y graficamos
media y varianza móviles. Un **Random Walk con Drift** exhibe una media móvil con pendiente
sistemática (tendencia direccional); un **Random Walk puro** vaga sin dirección
(media de las primeras diferencias ≈ 0 frente a su dispersión).

In [ ]:
serie = ener_noise["Ener_5"]
roll = stationarity.rolling_stats(serie, window=50)
diag = stationarity.classify_drift_vs_randomwalk(serie, window=50)

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax[0].plot(serie.index, serie.values, lw=0.7); ax[0].set_ylabel("Ener_5")
ax[0].set_title("T2 · Costo del Gas (Ener_5) — serie observada")
ax[1].plot(roll.index, roll["rolling_mean"], color="darkorange")
ax[1].set_ylabel("Media móvil (50)")
ax[2].plot(roll.index, roll["rolling_var"], color="seagreen")
ax[2].set_ylabel("Varianza móvil (50)"); ax[2].set_xlabel("tiempo (sintético)")
fig.tight_layout(); viz_utils.savefig(FIGS / "t2_ener5_rolling.png"); plt.show()

print("Diagnóstico Drift vs Random Walk para Ener_5:")
for k, v in diag.items():
    print(f"  {k}: {v}")

> **Lectura T2.** Si la media móvil de `Ener_5` muestra pendiente persistente y
> `drift_snr` supera el umbral, se clasifica como **Random Walk con Drift** (tendencia
> determinística embebida). La varianza móvil creciente confirmaría la no estacionariedad
> (heterocedasticidad), coherente con un factor macroeconómico I(1).